# Physics Competition Pipeline - EXACTS 2026

Notebook này chỉ tập trung vào phần **Physics QA**. Mục tiêu là thay pipeline cũ kiểu "LLM tự giải toàn bộ" bằng một pipeline ổn định hơn:

1. Load và audit dataset `verified_golden_expanded.csv`.
2. Giữ `holdout_test.csv` làm tập test thật, không dùng để train hay augment.
3. Route câu hỏi theo `topic` và `prefix`.
4. Retrieve ví dụ đúng dạng bằng vector database.
5. Dùng deterministic solver cho các dạng vật lý phổ biến.
6. Verify answer bằng code execution + unit-aware evaluator.
7. Chỉ dùng LLM/RAG fallback khi solver không đủ tự tin.
8. Chuẩn bị data SFT theo format sạch cho model <= 8B.

Tinh hoa giữ lại từ các notebook cũ:

- SciBERT/router idea: giữ tư tưởng route trước khi solve, nhưng route theo `topic` quan trọng hơn prefix.
- ChromaDB + sentence-transformers: giữ lại cho retrieval.
- Checklist vật lý: giữ lại nhưng đưa vào verifier/fallback prompt.
- Self-correction/self-consistency: giữ lại như fallback, không dùng mặc định cho mọi câu.
- Unit-aware evaluation: giữ lại và mở rộng.

# 1. Install Libraries

Cell này cài các thư viện nhẹ cần cho data, router, retrieval, evaluation.  
Các thư viện SFT/LLM nặng được để ở phần optional phía dưới để tránh Kaggle tự tải quá nhiều khi bạn chỉ muốn test pipeline.

In [ ]:
!pip install -q pandas numpy scikit-learn tqdm sentence-transformers chromadb

# 2. Imports And Global Config

Pipeline được viết để chạy được cả trên Kaggle lẫn local.  
Bạn chỉ cần chỉnh `DATA_SEARCH_ROOTS` nếu Kaggle input của bạn đặt tên khác.

In [ ]:
import ast
import csv
import json
import math
import os
import re
import shutil
import signal
import subprocess
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Switches for expensive parts.
USE_LLM_FALLBACK = False
RUN_SFT = False

# Prefer local Kaggle paths, but also work in this repository.
DATA_SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("."),
    Path(".."),
]

REQUIRED_COLUMNS = ["id", "prefix", "question", "golden_code", "cot", "answer", "unit", "topic"]

print("Ready.")

# 3. Locate Dataset Files

Dataset chính:

- `verified_golden_expanded.csv`: train/golden physics dataset.
- `holdout_test.csv`: test set đã để sẵn, không dùng để generate hoặc train.

Cell này tự tìm file theo tên. Nếu Kaggle input có nhiều bản trùng tên, hãy chỉnh lại path thủ công sau khi in danh sách candidate.

In [ ]:
def find_file(filename, roots=DATA_SEARCH_ROOTS):
    matches = []
    for root in roots:
        if not root.exists():
            continue
        matches.extend(root.rglob(filename))
    matches = sorted(set(matches), key=lambda p: (len(str(p)), str(p)))
    return matches

verified_candidates = find_file("verified_golden_expanded.csv")
holdout_candidates = find_file("holdout_test.csv")

print("verified_golden_expanded.csv candidates:")
for p in verified_candidates[:10]:
    print(" -", p)

print("\nholdout_test.csv candidates:")
for p in holdout_candidates[:10]:
    print(" -", p)

assert verified_candidates, "Cannot find verified_golden_expanded.csv"
assert holdout_candidates, "Cannot find holdout_test.csv"

VERIFIED_PATH = verified_candidates[0]
HOLDOUT_PATH = holdout_candidates[0]

print("\nUsing:")
print("VERIFIED_PATH =", VERIFIED_PATH)
print("HOLDOUT_PATH  =", HOLDOUT_PATH)

# 4. Load Data

Ta load dataset bằng pandas, ép các cột quan trọng về string để tránh mất ký hiệu khoa học, `Yes/No`, hoặc đơn vị.

In [ ]:
df = pd.read_csv(VERIFIED_PATH, dtype=str).fillna("")
holdout_df = pd.read_csv(HOLDOUT_PATH, dtype=str).fillna("")

for col in REQUIRED_COLUMNS:
    if col not in df.columns:
        raise ValueError(f"Missing required column in verified data: {col}")

print("verified rows:", len(df))
print("holdout rows :", len(holdout_df))
display(df.head(3))
display(holdout_df.head(3))

# 5. Dataset Audit

Audit cơ bản trước khi build pipeline:

- Đúng schema.
- Không duplicate id.
- Không overlap giữa train/golden và holdout.
- Không thiếu field quan trọng.
- `golden_code` parse được bằng Python AST.
- Phân phối topic/prefix.

In [ ]:
def audit_dataset(df, holdout_df):
    report = {}
    report["rows"] = len(df)
    report["holdout_rows"] = len(holdout_df)
    report["columns"] = list(df.columns)

    id_counts = Counter(df["id"])
    report["duplicate_ids"] = [k for k, v in id_counts.items() if v > 1]

    holdout_ids = set(holdout_df["id"]) if "id" in holdout_df.columns else set()
    report["holdout_overlap"] = sorted(set(df["id"]) & holdout_ids)

    missing = []
    for _, row in df.iterrows():
        for col in REQUIRED_COLUMNS:
            if not str(row.get(col, "")).strip():
                missing.append((row.get("id", ""), col))
    report["missing_required"] = missing

    syntax_errors = []
    for _, row in df.iterrows():
        try:
            ast.parse(str(row["golden_code"]))
        except Exception as e:
            syntax_errors.append((row["id"], str(e)))
    report["syntax_errors"] = syntax_errors

    report["prefix_counts"] = df["prefix"].value_counts().sort_index().to_dict()
    report["topic_counts"] = df["topic"].value_counts().sort_values().to_dict()
    return report

audit = audit_dataset(df, holdout_df)
print("rows:", audit["rows"])
print("holdout_rows:", audit["holdout_rows"])
print("duplicate_ids:", len(audit["duplicate_ids"]))
print("holdout_overlap:", len(audit["holdout_overlap"]))
print("missing_required:", len(audit["missing_required"]))
print("syntax_errors:", len(audit["syntax_errors"]))

print("\nPrefix counts:")
display(pd.Series(audit["prefix_counts"]).sort_values())

print("\nTopic counts:")
display(pd.Series(audit["topic_counts"]).sort_values())

assert len(audit["duplicate_ids"]) == 0
assert len(audit["holdout_overlap"]) == 0
assert len(audit["missing_required"]) == 0
assert len(audit["syntax_errors"]) == 0

# 6. Detect Constant/Template Code Rows

Một số dòng trong dataset có `golden_code` kiểu template hoặc gần như hardcode final answer.  
Các dòng này vẫn hữu ích cho answer/explanation SFT, nhưng không nên dùng làm target mạnh cho code generation.

In [ ]:
def looks_constant_code(code_text):
    text = str(code_text).strip()
    if not text:
        return True
    lower = text.lower()
    if "verified calculation target" in lower:
        return True
    if re.search(r"final_result\s*=\s*['\"]?[-+]?\d+(\.\d+)?", text):
        # This catches simple constants. It is a heuristic, not a proof.
        useful_ops = ["+", "-", "*", "/", "**", "math.", "sqrt", "sin", "cos"]
        before = text.split("final_result")[0]
        if not any(op in before for op in useful_ops):
            return True
    return False

df["code_quality"] = np.where(df["golden_code"].apply(looks_constant_code), "template_or_constant", "formula_code")
display(df["code_quality"].value_counts())
display(pd.crosstab(df["topic"], df["code_quality"]).sort_index())

# 7. Physics Normalization Utilities

Chuẩn hóa nhẹ text và đơn vị giúp router, retriever, solver ổn định hơn.  
Ta không rewrite câu hỏi gốc trong CSV, chỉ dùng normalized text cho model/pipeline.

In [ ]:
UNIT_ALIASES = {
    "ω": "ohm",
    "Ω": "ohm",
    "μ": "u",
    "µ": "u",
    "×": "x",
    "−": "-",
    "–": "-",
    "—": "-",
}

def normalize_text(text):
    text = str(text)
    for src, dst in UNIT_ALIASES.items():
        text = text.replace(src, dst)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_answer_text(text):
    text = normalize_text(text)
    text = text.replace("*10^", "e").replace("x10^", "e")
    return text

df["question_norm"] = df["question"].apply(normalize_text)
holdout_df["question_norm"] = holdout_df["question"].apply(normalize_text)

display(df[["id", "question", "question_norm"]].head(2))

# 8. Train Topic And Prefix Router

Ở test thật không có `topic`, nên cần một router.  
Router này dùng TF-IDF + Logistic Regression vì nhẹ, chạy nhanh trên Kaggle, và đủ tốt khi dataset đã có topic label sạch.

Lưu ý:

- `topic` là khóa solver chính.
- `prefix` là metadata phụ để retrieval chọn ví dụ đúng style.

In [ ]:
def train_router(label_col):
    train_df, valid_df = train_test_split(
        df,
        test_size=0.2,
        random_state=RANDOM_SEED,
        stratify=df[label_col],
    )
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=60000,
        )),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_SEED,
        )),
    ])
    pipe.fit(train_df["question_norm"], train_df[label_col])
    pred = pipe.predict(valid_df["question_norm"])
    print(f"\n=== {label_col} router ===")
    print("accuracy:", accuracy_score(valid_df[label_col], pred))
    print(classification_report(valid_df[label_col], pred))
    return pipe

topic_router = train_router("topic")
prefix_router = train_router("prefix")

# 9. Router Prediction With Confidence

Ta lấy xác suất cao nhất làm confidence.  
Khi confidence thấp, pipeline sẽ retrieve rộng hơn và có thể bật LLM fallback.

In [ ]:
def predict_with_confidence(pipe, texts):
    probs = pipe.predict_proba(texts)
    classes = pipe.named_steps["clf"].classes_
    idx = probs.argmax(axis=1)
    labels = classes[idx]
    conf = probs[np.arange(len(texts)), idx]
    return labels, conf

sample_texts = holdout_df["question_norm"].head(5).tolist()
topics, topic_conf = predict_with_confidence(topic_router, sample_texts)
prefixes, prefix_conf = predict_with_confidence(prefix_router, sample_texts)

pd.DataFrame({
    "question": sample_texts,
    "topic_pred": topics,
    "topic_conf": topic_conf,
    "prefix_pred": prefixes,
    "prefix_conf": prefix_conf,
})

# 10. Formula Cards

Formula cards là kiến thức cô đọng dùng cho retrieval và fallback prompt.  
Mỗi card chứa:

- topic
- công thức chính
- bẫy thường gặp
- unit convention

Đây là phần giúp model không phải tự nhớ toàn bộ vật lý từ prompt dài.

In [ ]:
FORMULA_CARDS = [
    {
        "topic": "circuit_power",
        "title": "AC/electric circuit power",
        "formulas": ["P = U*I", "P = I^2*R", "P = U^2/R at resonance or pure resistor", "P = U^2*R/Z^2"],
        "pitfalls": ["Use RMS voltage/current in AC problems.", "At resonance, series RLC impedance equals R."],
    },
    {
        "topic": "circuit_resistance",
        "title": "Resistance and AC impedance",
        "formulas": ["R = U/I", "R_series = R1 + R2", "1/R_parallel = 1/R1 + 1/R2", "Z = sqrt(R^2 + (XL-XC)^2)"],
        "pitfalls": ["When frequency is multiplied by n: XL' = n*XL and XC' = XC/n."],
    },
    {
        "topic": "measurement_error",
        "title": "Measurement error",
        "formulas": ["relative_error = absolute_error/value * 100%", "For product/quotient: relative errors add", "For sum/difference: absolute errors add"],
        "pitfalls": ["Keep percent as percent, not decimal fraction.", "Use absolute difference for absolute error."],
    },
    {
        "topic": "LC_oscillation",
        "title": "LC oscillation",
        "formulas": ["omega = 1/sqrt(L*C)", "f = 1/(2*pi*sqrt(L*C))", "T = 2*pi*sqrt(L*C)", "W = 0.5*C*U0^2", "Q0 = C*U0", "I0 = omega*Q0"],
        "pitfalls": ["Convert uF to F.", "Check whether answer asks angular frequency, frequency, or period."],
    },
    {
        "topic": "ac_resonance",
        "title": "RLC resonance",
        "formulas": ["XL = XC", "Z = R", "f0 = 1/(2*pi*sqrt(L*C))", "omega0 = 1/sqrt(L*C)"],
        "pitfalls": ["At resonance reactive voltages can be large, but total impedance is R."],
    },
    {
        "topic": "capacitor",
        "title": "Capacitor",
        "formulas": ["Q = C*U", "C = Q/U", "W = 0.5*C*U^2", "For disconnected capacitor: Q constant", "For connected capacitor: U constant"],
        "pitfalls": ["Convert uF, nF, uC correctly.", "Read whether source is disconnected or connected."],
    },
    {
        "topic": "electrostatics_force",
        "title": "Coulomb force",
        "formulas": ["F = k*abs(q1*q2)/r^2", "k = 9e9"],
        "pitfalls": ["Convert cm to m and uC/nC to C.", "For resultant force, handle vector direction."],
    },
    {
        "topic": "electrostatics_field",
        "title": "Electric field and potential",
        "formulas": ["E = k*abs(q)/r^2", "V = k*q/r", "E = U/d for uniform field"],
        "pitfalls": ["Field of positive charge points away; field of negative charge points toward."],
    },
    {
        "topic": "induction",
        "title": "Electromagnetic induction",
        "formulas": ["abs(e) = abs(delta_phi)/delta_t", "phi = B*S*cos(alpha)", "e = B*l*v for a moving rod"],
        "pitfalls": ["Use magnitude unless direction/sign is asked.", "Convert area and time units."],
    },
    {
        "topic": "general_physics",
        "title": "General physics",
        "formulas": ["Use direct formula extraction and unit conversion."],
        "pitfalls": ["This is a fallback bucket; prefer a specific topic when possible."],
    },
]

formula_df = pd.DataFrame(FORMULA_CARDS)
display(formula_df)

FORMULA_BY_TOPIC = {card["topic"]: card for card in FORMULA_CARDS}

# 11. Build Vector Database

Ta build ChromaDB từ toàn bộ `verified_golden_expanded.csv`, không dùng database cũ 156 câu.  
Document dùng cả question + topic + COT ngắn để embedding dễ tìm case tương tự.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

RAG_DIR = Path("/kaggle/working/physics_rag_database")
if RAG_DIR.exists():
    shutil.rmtree(RAG_DIR)
RAG_DIR.mkdir(parents=True, exist_ok=True)

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
client = chromadb.PersistentClient(path=str(RAG_DIR))
collection = client.get_or_create_collection(
    name="physics_verified_golden",
    embedding_function=embedding_fn,
)

documents = []
metadatas = []
ids = []

for _, row in df.iterrows():
    card = FORMULA_BY_TOPIC.get(row["topic"], {})
    formula_text = " ".join(card.get("formulas", []))
    doc = (
        f"TOPIC: {row['topic']}\n"
        f"PREFIX: {row['prefix']}\n"
        f"QUESTION: {row['question_norm']}\n"
        f"FORMULAS: {formula_text}\n"
        f"COT: {str(row['cot'])[:600]}"
    )
    documents.append(doc)
    metadatas.append({
        "id": row["id"],
        "prefix": row["prefix"],
        "topic": row["topic"],
        "question": row["question"],
        "golden_code": row["golden_code"],
        "cot": row["cot"],
        "answer": row["answer"],
        "unit": row["unit"],
        "code_quality": row["code_quality"],
    })
    ids.append(row["id"])

collection.add(documents=documents, metadatas=metadatas, ids=ids)
print("Chroma collection count:", collection.count())
print("Saved to:", RAG_DIR)

# 12. Retrieval Function

Retrieval ưu tiên cùng topic. Nếu topic confidence thấp hoặc không tìm đủ, fallback sang search toàn cục.

In [ ]:
def retrieve_examples(question, topic=None, prefix=None, k=4):
    question_norm = normalize_text(question)
    where = None
    if topic:
        where = {"topic": topic}

    try:
        result = collection.query(
            query_texts=[question_norm],
            n_results=k,
            where=where,
        )
    except Exception:
        result = collection.query(query_texts=[question_norm], n_results=k)

    examples = []
    for i in range(len(result["ids"][0])):
        meta = result["metadatas"][0][i]
        score = result.get("distances", [[None]])[0][i] if result.get("distances") else None
        examples.append({**meta, "distance": score})

    if len(examples) < k:
        result2 = collection.query(query_texts=[question_norm], n_results=k)
        seen = {ex["id"] for ex in examples}
        for i in range(len(result2["ids"][0])):
            meta = result2["metadatas"][0][i]
            if meta["id"] not in seen:
                examples.append({**meta, "distance": result2.get("distances", [[None]])[0][i]})
            if len(examples) >= k:
                break

    if prefix:
        examples = sorted(examples, key=lambda ex: 0 if ex["prefix"] == prefix else 1)
    return examples[:k]

sample_q = holdout_df.iloc[0]["question"]
sample_topic = predict_with_confidence(topic_router, [normalize_text(sample_q)])[0][0]
examples = retrieve_examples(sample_q, topic=sample_topic, k=3)
pd.DataFrame([{k: ex[k] for k in ["id", "prefix", "topic", "answer", "unit", "code_quality"]} for ex in examples])

# 13. Numeric And Unit-Aware Evaluation

Answer trong dataset có thể là:

- số thường: `5`
- số khoa học dạng text: `30.24 x 10^-3`
- string answer: `Yes`, `No`, hoặc mô tả ngắn.

Evaluator này tập trung vào numeric physics answer, có tolerance và unit scale cơ bản.

In [ ]:
UNIT_SCALE = {
    "-": 1.0,
    "": 1.0,
    "V": 1.0,
    "mV": 1e-3,
    "kV": 1e3,
    "A": 1.0,
    "mA": 1e-3,
    "uA": 1e-6,
    "C": 1.0,
    "uC": 1e-6,
    "nC": 1e-9,
    "F": 1.0,
    "uF": 1e-6,
    "nF": 1e-9,
    "H": 1.0,
    "mH": 1e-3,
    "ohm": 1.0,
    "Ω": 1.0,
    "N": 1.0,
    "J": 1.0,
    "W": 1.0,
    "Hz": 1.0,
    "rad/s": 1.0,
    "s": 1.0,
    "%": 1.0,
}

def parse_number(value):
    text = normalize_answer_text(value)
    text = text.replace(" ", "")
    text = re.sub(r"([0-9.]+)[xX]10\^?([-+]?\d+)", r"\1e\2", text)
    text = text.replace("^", "**")
    m = re.search(r"[-+]?\d+(?:\.\d+)?(?:e[-+]?\d+)?", text, flags=re.I)
    if not m:
        return None
    try:
        return float(m.group(0))
    except Exception:
        return None

def canonical_unit(unit):
    unit = str(unit).strip()
    unit = unit.replace("μ", "u").replace("µ", "u").replace("Ω", "ohm")
    return unit

def unit_scale(unit):
    u = canonical_unit(unit)
    return UNIT_SCALE.get(u, 1.0)

def compare_answer(pred_answer, pred_unit, true_answer, true_unit, rel_tol=2e-2, abs_tol=1e-6):
    p_num = parse_number(pred_answer)
    t_num = parse_number(true_answer)
    if p_num is None or t_num is None:
        return str(pred_answer).strip().lower() == str(true_answer).strip().lower()

    p_si = p_num * unit_scale(pred_unit)
    t_si = t_num * unit_scale(true_unit)
    return math.isclose(p_si, t_si, rel_tol=rel_tol, abs_tol=abs_tol)

print(compare_answer("0.004156", "N", "4.16 x 10^-3", "N"))
print(compare_answer("3.0", "mA", "0.003", "A"))

# 14. Safe Code Execution For Golden Code

Verifier có thể execute `golden_code` hoặc generated code trong môi trường giới hạn.  
Trong competition thật, hãy sandbox kỹ hơn; cell này đủ cho notebook/eval nội bộ.

In [ ]:
SAFE_BUILTINS = {
    "abs": abs,
    "round": round,
    "min": min,
    "max": max,
    "sum": sum,
    "pow": pow,
    "float": float,
    "int": int,
}

def execute_python_code(code_text, timeout_sec=3):
    code_text = str(code_text)
    tree = ast.parse(code_text)
    env = {
        "__builtins__": SAFE_BUILTINS,
        "math": math,
        "np": np,
    }
    loc = {}
    exec(compile(tree, "<physics_code>", "exec"), env, loc)
    if "final_result" not in loc:
        raise ValueError("Code did not set final_result")
    return loc["final_result"]

# Quick smoke test on a few formula-code rows.
for _, row in df[df["code_quality"] == "formula_code"].head(3).iterrows():
    try:
        got = execute_python_code(row["golden_code"])
        print(row["id"], "=>", got, "| target:", row["answer"], row["unit"])
    except Exception as e:
        print(row["id"], "execution issue:", e)

# 15. Variable Extraction Helpers

Các solver deterministic dùng regex đơn giản.  
Nguyên tắc: nếu không chắc pattern, solver trả `None` để fallback sang RAG/LLM thay vì đoán bừa.

In [ ]:
def find_value(text, names, unit=None):
    text_n = normalize_text(text)
    name_pat = "|".join(re.escape(n) for n in names)
    if unit:
        unit_pat = re.escape(unit).replace("ohm", "(?:ohm|Ω)")
        pat = rf"(?:{name_pat})\s*=\s*([-+]?\d+(?:\.\d+)?)\s*{unit_pat}"
    else:
        pat = rf"(?:{name_pat})\s*=\s*([-+]?\d+(?:\.\d+)?)"
    m = re.search(pat, text_n, flags=re.I)
    return float(m.group(1)) if m else None

def find_all_numbers_with_unit(text, unit):
    text_n = normalize_text(text)
    unit_pat = re.escape(unit).replace("ohm", "(?:ohm|Ω)")
    return [float(x) for x in re.findall(rf"([-+]?\d+(?:\.\d+)?)\s*{unit_pat}", text_n, flags=re.I)]

def unit_to_si(value, unit):
    unit = canonical_unit(unit)
    return value * UNIT_SCALE.get(unit, 1.0)

@dataclass
class SolverResult:
    answer: str
    unit: str
    explanation: str
    topic: str
    confidence: float
    method: str
    code: str = ""

def result(answer, unit, explanation, topic, confidence, method, code=""):
    if isinstance(answer, float):
        answer = f"{answer:.6g}"
    return SolverResult(str(answer), unit, explanation, topic, confidence, method, code)

# 16. Deterministic Solvers

Các solver này cover những dạng physics phổ biến nhất trong dataset:

- circuit_power
- circuit_resistance
- measurement_error
- LC_oscillation
- ac_resonance
- capacitor
- electrostatics_force
- electrostatics_field
- induction

Không cần solver nào cũng bắt mọi câu. Chỉ cần solver trả lời khi pattern rõ ràng và tự tin.

In [ ]:
def solve_circuit_power(question):
    q = normalize_text(question)
    U = find_value(q, ["U", "voltage"], "V")
    I = find_value(q, ["I", "current"], "A")
    R = find_value(q, ["R", "resistance"], "ohm")
    Z = find_value(q, ["Z", "impedance"], "ohm")

    if U is not None and R is not None and ("resonance" in q.lower() or "resistor" in q.lower()) and "Z" not in q:
        ans = U**2 / R
        code = f"U={U}\nR={R}\nfinal_result=U**2/R"
        exp = f"At resonance or for a pure resistor, P=U^2/R. Substitute U={U} V and R={R} ohm, so P={ans:.6g} W."
        return result(ans, "W", exp, "circuit_power", 0.92, "deterministic", code)

    if U is not None and I is not None:
        ans = U * I
        code = f"U={U}\nI={I}\nfinal_result=U*I"
        exp = f"Use P=UI. Substitute U={U} V and I={I} A, so P={ans:.6g} W."
        return result(ans, "W", exp, "circuit_power", 0.95, "deterministic", code)

    if I is not None and R is not None:
        ans = I**2 * R
        code = f"I={I}\nR={R}\nfinal_result=I**2*R"
        exp = f"Use Joule power P=I^2R. Substitute I={I} A and R={R} ohm, so P={ans:.6g} W."
        return result(ans, "W", exp, "circuit_power", 0.95, "deterministic", code)

    if U is not None and R is not None and Z is not None:
        ans = U**2 * R / Z**2
        code = f"U={U}\nR={R}\nZ={Z}\nI=U/Z\nfinal_result=I**2*R"
        exp = f"In an AC circuit, I=U/Z and active power is P=I^2R. Thus P=({U}/{Z})^2*{R}={ans:.6g} W."
        return result(ans, "W", exp, "circuit_power", 0.9, "deterministic", code)

    return None


def solve_circuit_resistance(question):
    q = normalize_text(question)
    U = find_value(q, ["U", "voltage"], "V")
    I = find_value(q, ["I", "current"], "A")
    R = find_value(q, ["R", "resistance"], "ohm")
    XL = find_value(q, ["XL", "X_L"], "ohm")
    XC = find_value(q, ["XC", "X_C"], "ohm")

    if U is not None and I is not None and ("resistance" in q.lower() or "ohm" in q.lower()):
        ans = U / I
        code = f"U={U}\nI={I}\nfinal_result=U/I"
        exp = f"By Ohm's law, R=U/I={U}/{I}={ans:.6g} ohm."
        return result(ans, "Ω", exp, "circuit_resistance", 0.95, "deterministic", code)

    nums_ohm = find_all_numbers_with_unit(q, "ohm")
    if "series" in q.lower() and len(nums_ohm) >= 2:
        ans = sum(nums_ohm[:2])
        code = f"R1={nums_ohm[0]}\nR2={nums_ohm[1]}\nfinal_result=R1+R2"
        exp = f"For series resistors, equivalent resistance is the sum: {nums_ohm[0]}+{nums_ohm[1]}={ans:.6g} ohm."
        return result(ans, "Ω", exp, "circuit_resistance", 0.92, "deterministic", code)

    if "parallel" in q.lower() and len(nums_ohm) >= 2:
        r1, r2 = nums_ohm[:2]
        ans = r1 * r2 / (r1 + r2)
        code = f"R1={r1}\nR2={r2}\nfinal_result=R1*R2/(R1+R2)"
        exp = f"For two parallel resistors, R_eq=R1R2/(R1+R2)={ans:.6g} ohm."
        return result(ans, "Ω", exp, "circuit_resistance", 0.92, "deterministic", code)

    factor_match = re.search(r"(?:increased|tripled|quadrupled|multiplied).*?(\d+(?:\.\d+)?)\s*times", q, flags=re.I)
    if not factor_match:
        word_factor = {"tripled": 3, "quadrupled": 4, "doubled": 2}
        for word, val in word_factor.items():
            if word in q.lower():
                factor_match = type("M", (), {"group": lambda self, _: str(val)})()
                break

    if XL is not None and XC is not None and factor_match is not None and U is not None:
        n = float(factor_match.group(1))
        XL_new = n * XL
        XC_new = XC / n
        if abs(XL_new - XC_new) < 1e-9:
            if R is not None and "current" in q.lower():
                ans = U / R
                unit = "A"
                exp = f"After frequency changes by {n:g}, XL'={XL_new:g} and XC'={XC_new:g}, so resonance occurs. Then Z=R={R:g} ohm and I=U/R={ans:.6g} A."
            else:
                ans = U
                unit = "V"
                exp = f"After frequency changes by {n:g}, XL'={XL_new:g} and XC'={XC_new:g}, so resonance occurs. The resistor voltage equals source voltage: {ans:.6g} V."
            code = f"XL={XL}\nXC={XC}\nn={n}\nU={U}\nR={R if R is not None else 0}\nfinal_result={ans}"
            return result(ans, unit, exp, "circuit_resistance", 0.9, "deterministic", code)

    return None


def solve_measurement_error(question):
    q = normalize_text(question)
    q_lower = q.lower()
    pm = re.search(r"([-+]?\d+(?:\.\d+)?)\s*(?:±|\+/-|\+-)\s*([-+]?\d+(?:\.\d+)?)", q)
    if pm and ("maximum" in q_lower or "minimum" in q_lower):
        value = float(pm.group(1))
        delta = float(pm.group(2))
        ans = value + delta if "maximum" in q_lower else value - delta
        word = "maximum" if "maximum" in q_lower else "minimum"
        exp = f"The {word} possible value is measured value {'+' if word == 'maximum' else '-'} uncertainty = {ans:.6g}."
        return result(ans, "-", exp, "measurement_error", 0.93, "deterministic")

    if "relative error" in q_lower:
        nums = [float(x) for x in re.findall(r"[-+]?\d+(?:\.\d+)?", q)]
        if len(nums) >= 2:
            # Prefer smaller value as uncertainty when phrase is simple.
            delta, value = sorted(nums[:2])
            ans = delta / value * 100
            exp = f"Relative error is delta/value*100% = {delta}/{value}*100% = {ans:.6g}%."
            return result(ans, "%", exp, "measurement_error", 0.85, "deterministic")

    if "absolute error" in q_lower and "accepted" in q_lower:
        nums = [float(x) for x in re.findall(r"[-+]?\d+(?:\.\d+)?", q)]
        if len(nums) >= 2:
            ans = abs(nums[0] - nums[1])
            exp = f"Absolute error is |measured-accepted| = |{nums[0]}-{nums[1]}| = {ans:.6g}."
            return result(ans, "-", exp, "measurement_error", 0.9, "deterministic")

    return None


def solve_lc_oscillation(question):
    q = normalize_text(question)
    q_lower = q.lower()
    L = find_value(q, ["L", "inductance"], "H")
    C_uf = find_value(q, ["C", "capacitance"], "uF")
    U0 = find_value(q, ["U0", "maximum voltage", "voltage"], "V")
    if L is None or C_uf is None:
        return None
    C = C_uf * 1e-6

    if "angular" in q_lower or "omega" in q_lower:
        ans = 1 / math.sqrt(L * C)
        exp = f"Convert C={C_uf} uF to {C:g} F. Angular frequency omega=1/sqrt(LC)={ans:.6g} rad/s."
        return result(ans, "rad/s", exp, "LC_oscillation", 0.95, "deterministic")
    if "frequency" in q_lower:
        ans = 1 / (2 * math.pi * math.sqrt(L * C))
        exp = f"Convert C={C_uf} uF to {C:g} F. Frequency f=1/(2*pi*sqrt(LC))={ans:.6g} Hz."
        return result(ans, "Hz", exp, "LC_oscillation", 0.95, "deterministic")
    if "period" in q_lower:
        ans = 2 * math.pi * math.sqrt(L * C)
        exp = f"Convert C={C_uf} uF to {C:g} F. Period T=2*pi*sqrt(LC)={ans:.6g} s."
        return result(ans, "s", exp, "LC_oscillation", 0.95, "deterministic")
    if U0 is not None and "charge" in q_lower:
        ans = C * U0
        exp = f"Maximum charge Q0=CU0={C:g}*{U0}={ans:.6g} C."
        return result(ans, "C", exp, "LC_oscillation", 0.92, "deterministic")
    if U0 is not None and "energy" in q_lower:
        ans = 0.5 * C * U0**2
        exp = f"Total energy W=0.5*C*U0^2=0.5*{C:g}*{U0}^2={ans:.6g} J."
        return result(ans, "J", exp, "LC_oscillation", 0.92, "deterministic")
    return None


def solve_capacitor(question):
    q = normalize_text(question)
    q_lower = q.lower()
    C_uf = find_value(q, ["C", "capacitance"], "uF")
    U = find_value(q, ["U", "voltage"], "V")
    Q_uc = find_value(q, ["Q", "charge"], "uC")

    if C_uf is not None and U is not None and "energy" in q_lower:
        C = C_uf * 1e-6
        ans = 0.5 * C * U**2
        exp = f"Capacitor energy W=0.5*C*U^2 with C={C_uf} uF={C:g} F and U={U} V, so W={ans:.6g} J."
        return result(ans, "J", exp, "capacitor", 0.92, "deterministic")
    if C_uf is not None and U is not None:
        ans = C_uf * U
        exp = f"Charge Q=CU. Using C in uF gives Q in uC: Q={C_uf}*{U}={ans:.6g} uC."
        return result(ans, "μC", exp, "capacitor", 0.9, "deterministic")
    if Q_uc is not None and U is not None:
        ans = Q_uc / U
        exp = f"Capacitance C=Q/U={Q_uc}/{U}={ans:.6g} uF."
        return result(ans, "μF", exp, "capacitor", 0.9, "deterministic")
    return None


def solve_electrostatics(question, topic):
    q = normalize_text(question)
    q_lower = q.lower()
    # Direct single-source formulas only; vector cases should usually go to fallback.
    k = 9e9
    q_uc = find_value(q, ["q", "charge"], "uC")
    q_nc = find_value(q, ["q", "charge"], "nC")
    r_cm = find_value(q, ["r", "distance"], "cm")
    if q_uc is not None:
        charge = q_uc * 1e-6
    elif q_nc is not None:
        charge = q_nc * 1e-9
    else:
        charge = None
    r = r_cm * 1e-2 if r_cm is not None else None

    if charge is not None and r is not None and ("field" in q_lower or topic == "electrostatics_field"):
        ans = k * abs(charge) / r**2
        exp = f"Electric field magnitude of a point charge is E=k|q|/r^2. Convert q={charge:g} C and r={r:g} m, so E={ans:.6g} V/m."
        return result(ans, "V/m", exp, "electrostatics_field", 0.82, "deterministic")
    return None


def solve_induction(question):
    q = normalize_text(question)
    q_lower = q.lower()
    nums = [float(x) for x in re.findall(r"[-+]?\d+(?:\.\d+)?", q)]
    if "flux" in q_lower and "time" in q_lower and len(nums) >= 2:
        delta_phi, delta_t = nums[0], nums[1]
        ans = abs(delta_phi / delta_t)
        exp = f"Induced emf magnitude is |delta_phi|/delta_t = |{delta_phi}|/{delta_t} = {ans:.6g} V."
        return result(ans, "V", exp, "induction", 0.75, "deterministic")
    return None


SOLVERS = {
    "circuit_power": solve_circuit_power,
    "circuit_resistance": solve_circuit_resistance,
    "measurement_error": solve_measurement_error,
    "LC_oscillation": solve_lc_oscillation,
    "ac_resonance": solve_circuit_resistance,
    "capacitor": solve_capacitor,
    "electrostatics_force": lambda q: solve_electrostatics(q, "electrostatics_force"),
    "electrostatics_field": lambda q: solve_electrostatics(q, "electrostatics_field"),
    "induction": solve_induction,
}

# 17. Checklist For Fallback Prompt

Checklist này lấy từ notebook cũ nhưng dùng đúng vị trí hơn: đưa vào prompt fallback hoặc verifier, không bắt LLM nhớ mọi thứ trong lần solve đầu.

In [ ]:
UNIVERSAL_CHECKLIST = '''
UNIVERSAL PHYSICS CHECKS:
1. Convert all quantities to SI before computing unless the requested unit is explicitly non-SI.
2. Constants: k=9e9, g=10 when high-school convention is implied.
3. For RLC resonance: XL=XC and Z=R.
4. When frequency is multiplied by n: XL'=n*XL and XC'=XC/n.
5. Capacitor: disconnected source means Q constant; connected source means U constant.
6. Coulomb force: F=k*|q1*q2|/r^2; electric field: E=k*|q|/r^2.
7. For measurement products/quotients, relative errors add.
8. Output must contain final answer and unit.
'''.strip()
print(UNIVERSAL_CHECKLIST)

# 18. Optional LLM Fallback

Phần này giữ lại tinh hoa RAG code-generation từ notebook cũ, nhưng chỉ dùng khi:

- deterministic solver không cover,
- router confidence thấp,
- hoặc verifier fail.

Mặc định `USE_LLM_FALLBACK = False`. Bật lên nếu Kaggle runtime đã có Ollama hoặc bạn đã setup model local/open-source.

In [ ]:
def call_ollama(prompt, model_name="qwen2.5-math:7b", temperature=0.1, timeout=180):
    import requests
    url = "http://localhost:11434/api/chat"
    payload = {
        "model": model_name,
        "messages": [{"role": "user", "content": prompt}],
        "options": {"temperature": temperature},
        "stream": False,
    }
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()["message"]["content"]

def build_fallback_prompt(question, topic, prefix, examples):
    card = FORMULA_BY_TOPIC.get(topic, {})
    ex_texts = []
    for ex in examples:
        ex_texts.append(
            f"ID: {ex['id']}\n"
            f"TOPIC: {ex['topic']}\n"
            f"QUESTION: {ex['question']}\n"
            f"COT: {ex['cot']}\n"
            f"ANSWER: {ex['answer']} {ex['unit']}\n"
            f"CODE:\n{ex['golden_code']}"
        )
    prompt = f'''
You are a physics solver for an educational QA benchmark.

Predicted topic: {topic}
Predicted prefix: {prefix}

Formula card:
{json.dumps(card, ensure_ascii=False, indent=2)}

Checklist:
{UNIVERSAL_CHECKLIST}

Retrieved examples:
{chr(10).join(ex_texts)}

New question:
{question}

Return JSON only with:
{{
  "answer": "...",
  "unit": "...",
  "explanation": "...",
  "python_code": "code that sets final_result"
}}
'''
    return prompt.strip()

def parse_json_from_text(text):
    text = text.strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def llm_fallback_solve(question, topic, prefix, examples):
    if not USE_LLM_FALLBACK:
        return None
    prompt = build_fallback_prompt(question, topic, prefix, examples)
    raw = call_ollama(prompt)
    obj = parse_json_from_text(raw)
    if not obj:
        return None
    return result(
        obj.get("answer", ""),
        obj.get("unit", "-"),
        obj.get("explanation", raw[:1000]),
        topic,
        0.55,
        "llm_rag_fallback",
        obj.get("python_code", ""),
    )

# 19. Unified Physics Solver

Đây là entrypoint chính:

1. Normalize question.
2. Predict topic/prefix.
3. Retrieve examples.
4. Try deterministic solver.
5. Verify against known answer nếu đang eval.
6. Fallback LLM/RAG nếu cần.
7. Return answer + unit + explanation + metadata.

In [ ]:
def solve_physics_question(question, known_answer=None, known_unit=None):
    q_norm = normalize_text(question)
    topic_arr, topic_conf_arr = predict_with_confidence(topic_router, [q_norm])
    prefix_arr, prefix_conf_arr = predict_with_confidence(prefix_router, [q_norm])
    topic = topic_arr[0]
    prefix = prefix_arr[0]
    topic_conf = float(topic_conf_arr[0])
    prefix_conf = float(prefix_conf_arr[0])

    examples = retrieve_examples(question, topic=topic, prefix=prefix, k=4)

    solver = SOLVERS.get(topic)
    sol = solver(question) if solver else None

    verified = None
    if sol and known_answer is not None:
        verified = compare_answer(sol.answer, sol.unit, known_answer, known_unit)
        if not verified and USE_LLM_FALLBACK:
            sol = llm_fallback_solve(question, topic, prefix, examples)
    elif sol:
        verified = None

    if sol is None:
        sol = llm_fallback_solve(question, topic, prefix, examples)

    if sol is None:
        # Retrieval-only conservative fallback: do not pretend this is solved.
        nearest = examples[0] if examples else {}
        sol = result(
            answer=nearest.get("answer", ""),
            unit=nearest.get("unit", "-"),
            explanation=(
                "No deterministic solver matched confidently. "
                "Returned nearest retrieved example answer as a low-confidence fallback; "
                "enable LLM fallback or add a solver for this pattern."
            ),
            topic=topic,
            confidence=0.2,
            method="retrieval_low_confidence",
            code=nearest.get("golden_code", ""),
        )
        if known_answer is not None:
            verified = compare_answer(sol.answer, sol.unit, known_answer, known_unit)

    return {
        "answer": sol.answer,
        "unit": sol.unit,
        "explanation": sol.explanation,
        "topic_pred": topic,
        "topic_conf": topic_conf,
        "prefix_pred": prefix,
        "prefix_conf": prefix_conf,
        "solver_conf": sol.confidence,
        "method": sol.method,
        "verified_if_known": verified,
        "retrieved_ids": [ex["id"] for ex in examples],
        "python_code": sol.code,
    }

demo = solve_physics_question(holdout_df.iloc[0]["question"], holdout_df.iloc[0]["answer"], holdout_df.iloc[0]["unit"])
json.dumps(demo, ensure_ascii=False, indent=2)

# 20. Evaluate On `holdout_test.csv`

Đây là evaluation thật của pipeline hiện tại.  
Vì deterministic solvers chưa cover 100% mọi dạng, hãy đọc `method`:

- `deterministic`: solver công thức tự tin.
- `retrieval_low_confidence`: chưa solve thật, cần bổ sung solver hoặc bật LLM fallback.
- `llm_rag_fallback`: dùng model fallback nếu bạn bật `USE_LLM_FALLBACK`.

In [ ]:
eval_rows = []
for _, row in tqdm(holdout_df.iterrows(), total=len(holdout_df)):
    out = solve_physics_question(
        row["question"],
        known_answer=row.get("answer", ""),
        known_unit=row.get("unit", "-"),
    )
    eval_rows.append({
        "id": row.get("id", ""),
        "question": row.get("question", ""),
        "true_answer": row.get("answer", ""),
        "true_unit": row.get("unit", "-"),
        **out,
    })

eval_df = pd.DataFrame(eval_rows)
eval_df["is_correct"] = eval_df["verified_if_known"].fillna(False).astype(bool)

print("Holdout accuracy:", eval_df["is_correct"].mean())
display(eval_df["method"].value_counts())
display(pd.crosstab(eval_df["topic_pred"], eval_df["is_correct"]))
display(eval_df.head(10))

EVAL_REPORT_PATH = Path("/kaggle/working/physics_holdout_pipeline_eval.csv")
eval_df.to_csv(EVAL_REPORT_PATH, index=False)
print("Saved:", EVAL_REPORT_PATH)

# 21. Error Analysis

Nhìn các câu sai theo topic/method để quyết định:

- thêm solver pattern,
- sửa router,
- bổ sung formula card,
- hoặc bật LLM fallback cho nhóm đó.

In [ ]:
wrong_df = eval_df[~eval_df["is_correct"]].copy()
print("Wrong rows:", len(wrong_df))
display(wrong_df[[
    "id", "topic_pred", "topic_conf", "prefix_pred", "method",
    "answer", "unit", "true_answer", "true_unit", "retrieved_ids", "question"
]].head(30))

wrong_path = Path("/kaggle/working/physics_holdout_errors.csv")
wrong_df.to_csv(wrong_path, index=False)
print("Saved:", wrong_path)

# 22. Build SFT Dataset

SFT nên dạy model output ổn định, không chỉ chain-of-thought dài.  
Format đề xuất:

```json
{
  "topic": "...",
  "answer": "...",
  "unit": "...",
  "explanation": "...",
  "python_code": "..."
}
```

Quan trọng:

- `formula_code` rows: dùng được cho `python_code`.
- `template_or_constant` rows: dùng cho answer/explanation, không nên ép model học code hardcode.

In [ ]:
def make_sft_record(row):
    use_code = row["code_quality"] == "formula_code"
    target = {
        "topic": row["topic"],
        "answer": row["answer"],
        "unit": row["unit"],
        "explanation": row["cot"],
    }
    if use_code:
        target["python_code"] = row["golden_code"]
    else:
        target["python_code"] = ""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a physics educational QA solver. "
                "Return correct answer, unit, and a clear explanation. "
                "Use Python code only when it is formula-based, not hardcoded."
            ),
        },
        {
            "role": "user",
            "content": row["question"],
        },
        {
            "role": "assistant",
            "content": json.dumps(target, ensure_ascii=False),
        },
    ]
    return {
        "id": row["id"],
        "topic": row["topic"],
        "prefix": row["prefix"],
        "code_quality": row["code_quality"],
        "messages": messages,
    }

sft_records = [make_sft_record(row) for _, row in df.iterrows()]
sft_train, sft_valid = train_test_split(
    sft_records,
    test_size=0.08,
    random_state=RANDOM_SEED,
    stratify=[r["topic"] for r in sft_records],
)

SFT_DIR = Path("/kaggle/working/physics_sft_data")
SFT_DIR.mkdir(parents=True, exist_ok=True)
train_path = SFT_DIR / "train.jsonl"
valid_path = SFT_DIR / "valid.jsonl"

with train_path.open("w", encoding="utf-8") as f:
    for r in sft_train:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
with valid_path.open("w", encoding="utf-8") as f:
    for r in sft_valid:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("train:", len(sft_train), train_path)
print("valid:", len(sft_valid), valid_path)
display(pd.Series([r["code_quality"] for r in sft_records]).value_counts())

# 23. Optional QLoRA SFT

Phần này optional vì cần GPU và thời gian.  
Model gợi ý trong phạm vi <= 8B:

- `Qwen/Qwen2.5-Math-7B-Instruct`: mạnh về math/code reasoning.
- `Qwen/Qwen3-8B`: hợp hơn nếu bạn muốn một model cho cả logic + physics.

Bật `RUN_SFT = True` nếu muốn train.

In [ ]:
if RUN_SFT:
    import sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers", "accelerate", "peft", "trl", "bitsandbytes", "datasets"
    ])
else:
    print("RUN_SFT is False. Skip installing heavy SFT libraries.")

In [ ]:
if RUN_SFT:
    import torch
    from datasets import load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig
    from trl import SFTTrainer, SFTConfig

    BASE_MODEL = "Qwen/Qwen2.5-Math-7B-Instruct"
    OUTPUT_DIR = "/kaggle/working/qwen_physics_lora"

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    dataset = load_dataset("json", data_files={
        "train": str(train_path),
        "validation": str(valid_path),
    })

    def format_messages(example):
        return tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )

    sft_config = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=100,
        save_total_limit=2,
        bf16=True,
        max_seq_length=2048,
        packing=False,
        dataset_text_field="text",
    )

    dataset = dataset.map(lambda ex: {"text": format_messages(ex)})

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        args=sft_config,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        peft_config=lora_config,
    )

    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Saved LoRA adapter to:", OUTPUT_DIR)
else:
    print("RUN_SFT is False. SFT cell skipped.")

# 24. Optional Self-Consistency Fallback

Self-consistency từ notebook cũ có ích, nhưng không nên dùng cho mọi câu vì tốn thời gian.  
Chỉ dùng khi:

- deterministic solver fail,
- router confidence thấp,
- hoặc output LLM không verify được.

In [ ]:
def self_consistency_llm(question, topic, prefix, n=3):
    if not USE_LLM_FALLBACK:
        return None
    examples = retrieve_examples(question, topic=topic, prefix=prefix, k=4)
    candidates = []
    for temp in [0.1, 0.4, 0.7][:n]:
        sol = llm_fallback_solve(question, topic, prefix, examples)
        if sol:
            candidates.append(sol)
    if not candidates:
        return None
    grouped = Counter((c.answer, c.unit) for c in candidates)
    (ans, unit), _ = grouped.most_common(1)[0]
    chosen = next(c for c in candidates if c.answer == ans and c.unit == unit)
    return chosen

# 25. Competition-Style API Function

Đây là wrapper cuối cùng bạn có thể đưa vào inference script/API.  
Nó trả về answer + explanation, và metadata debug nếu bạn cần.

In [ ]:
def answer_physics_api(question, debug=False):
    out = solve_physics_question(question)
    response = {
        "answer": out["answer"],
        "unit": out["unit"],
        "explanation": out["explanation"],
    }
    if debug:
        response["debug"] = {
            "topic_pred": out["topic_pred"],
            "topic_conf": out["topic_conf"],
            "prefix_pred": out["prefix_pred"],
            "prefix_conf": out["prefix_conf"],
            "method": out["method"],
            "retrieved_ids": out["retrieved_ids"],
        }
    return response

test_question = holdout_df.iloc[0]["question"]
json.dumps(answer_physics_api(test_question, debug=True), ensure_ascii=False, indent=2)

# 26. What To Improve Next

Sau khi chạy notebook này, thứ tự cải thiện nên là:

1. Mở `physics_holdout_errors.csv`.
2. Nhóm lỗi theo `topic_pred` và `method`.
3. Với lỗi `retrieval_low_confidence`, thêm deterministic solver pattern trước.
4. Với lỗi router sai topic, tăng data hoặc đổi router sang SciBERT/Qwen embedding classifier.
5. Với lỗi unit scale, bổ sung `UNIT_SCALE` và parser.
6. Với lỗi explanation tốt nhưng answer sai, ép pipeline lấy answer từ solver/verifier, không lấy từ LLM.
7. Chỉ fine-tune sau khi evaluation script ổn định, vì SFT trên output chưa verify sẽ khuếch đại lỗi.